Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [1]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [2]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [3]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [4]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [5]:
# windowing function
def create_windowed_data(data, past, future):
    X, y = [], []
    for i in range(len(data) - past - future + 1):
        X.append(data[i:i + past])
        y.append(data[i + past + future - 1])  # Solo la temperatura como etiqueta
    return np.array(X), np.array(y)

In [6]:
past, future = (5, 2)

X_train, y_train = create_windowed_data(train_data, past, future)
X_test, y_test = create_windowed_data(test_data, past, future)

print('X_train:', X_train[:4])
print('y_train:', y_train[:4])

X_train: [[20.7 17.9 18.8 14.6 15.8]
 [17.9 18.8 14.6 15.8 15.8]
 [18.8 14.6 15.8 15.8 15.8]
 [14.6 15.8 15.8 15.8 17.4]]
y_train: [15.8 17.4 21.8 20. ]


<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [7]:
inputs = keras.layers.Input(shape=(past,1))

# Crear las capas GRU
x = layers.GRU(64, activation='relu', return_sequences=True)(inputs)
x = layers.GRU(32, activation='relu')(x)

output = layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 5, 64)          │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,305 (87.13 KB)

 Trainable params: 22,305 (87.13 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - loss: 98.8803 - val_loss: 9.5868
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 9.7685 - val_loss: 9.1557
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.5082 - val_loss: 8.8022
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.0191 - val_loss: 9.0816
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.3013 - val_loss: 8.8718
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.7092 - val_loss: 8.9249
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.6145 - val_loss: 8.4331
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.8873 - val_loss: 8.4205
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.7586 - val_loss: 8.3897
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 8.3589 - val_loss: 8.4422
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.2026 - val_loss: 8.3341
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss:

In [9]:

results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 6.7180
Test Loss: 7.10966682434082


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [10]:
# Creo nuevas características
df['portion_year'] = df['Date'].dt.dayofyear / 365.0
df['Month'] = df['Date'].dt.month  # Mes del año
df['DayOfYear'] = df['Date'].dt.dayofyear  # Día del año
df['Season'] = df['Month'].apply(lambda x: (x%12 + 3)//3)  # Estaciones del año
df['DaysSinceStart'] = (df['Date'] - df['Date'].min()).dt.days  # Tendencia temporal

# Creo el DataFrame con las características
df_multi = df[['Temp', 'portion_year', 'Month', 'DayOfYear', 'Season', 'DaysSinceStart']].copy()

# Normalizo DF
scaler = MinMaxScaler((0,1))
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_multi),
    columns=df_multi.columns
)

train_data = df_scaled.iloc[:3000].values
test_data  = df_scaled.iloc[3000:].values

In [11]:
X_train, y_train = create_windowed_data(train_data, past, future)
X_test, y_test = create_windowed_data(test_data, past, future)

print(f'X_train shape: {X_train.shape}, y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}, y_test shape: {y_test.shape}')

X_train shape: (2994, 5, 6), y_train shape: (2994, 6)
X_test shape: (644, 5, 6), y_test shape: (644, 6)


In [12]:
inputs = layers.Input(shape=(past, 6))
x = layers.GRU(64, return_sequences=True)(inputs)
x = layers.GRU(32)(x)
x = layers.Dropout(0.2)(x)
output = layers.Dense(1)(x)

model = models.Model(inputs, output)
optimizer = Adam(learning_rate=1e-3, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='mean_squared_error')

In [13]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 0.1092 - val_loss: 0.0552
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0531 - val_loss: 0.0554
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0502 - val_loss: 0.0553
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0507 - val_loss: 0.0552
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0488 - val_loss: 0.0553
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0490 - val_loss: 0.0548
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0491 - val_loss: 0.0557
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0494 - val_loss: 0.0549
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0478 - val_loss: 0.0546
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0481 - val_loss: 0.0546
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0497 - val_loss: 0.0545
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 

In [14]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0519
Test Loss: 0.060089047998189926


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


Las respuestas correctas son **a)**, **b)** y **d)**.
La c) es incorrecta porque seria many-to-many

<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


***Todas las respuestas son correctas.***